# 24 · Triggers & Advanced Views

- **Triggers** run SQL automatically in response to `INSERT`/`UPDATE`/`DELETE`.
  Uses: audit logs, derived columns, validation, enforcing rules.
- **`INSTEAD OF` triggers** make a view **updatable**.

Triggers reference the special rows `NEW` (the incoming row) and `OLD` (the prior
row). Everything here uses throwaway `demo_` objects.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Set up an accounts table and an audit log

In [ ]:
%%sql
DROP TRIGGER IF EXISTS demo_bal_audit;
DROP TABLE IF EXISTS demo_accounts;
DROP TABLE IF EXISTS demo_audit;

CREATE TABLE demo_accounts (id INTEGER PRIMARY KEY, name TEXT, balance REAL);
CREATE TABLE demo_audit    (entry_id INTEGER PRIMARY KEY, note TEXT);
INSERT INTO demo_accounts VALUES (1, 'Alice', 100.0), (2, 'Bob', 50.0);
SELECT 'setup done' AS status;

## `AFTER UPDATE` trigger → audit log
Whenever a balance changes, automatically record the change. `AFTER UPDATE OF
balance` fires only when that column is updated.

In [ ]:
%%sql
CREATE TRIGGER demo_bal_audit
AFTER UPDATE OF balance ON demo_accounts
BEGIN
    INSERT INTO demo_audit (note)
    VALUES (NEW.name || ': ' || OLD.balance || ' -> ' || NEW.balance);
END;

Now update a balance and watch the audit row appear by itself:

In [ ]:
%%sql
UPDATE demo_accounts SET balance = balance + 25 WHERE id = 1;
UPDATE demo_accounts SET balance = balance - 10 WHERE id = 2;
SELECT * FROM demo_audit;

## `BEFORE` trigger for validation with `RAISE`
Reject invalid changes before they happen. This trigger aborts any update that
would make a balance negative.

In [ ]:
%%sql
DROP TRIGGER IF EXISTS demo_no_negative;
CREATE TRIGGER demo_no_negative
BEFORE UPDATE OF balance ON demo_accounts
WHEN NEW.balance < 0
BEGIN
    SELECT RAISE(ABORT, 'balance cannot go negative');
END;
SELECT 'validation trigger created' AS status;

This next update **should fail** on purpose (Bob can't go to -1000), and the audit table stays clean because the update never commits:

In [ ]:
%%sql
UPDATE demo_accounts SET balance = balance - 1000 WHERE id = 2;

## Updatable view via `INSTEAD OF`
Views are read-only by default. An `INSTEAD OF` trigger intercepts writes to the
view and applies them to the underlying table.

In [ ]:
%%sql
DROP VIEW IF EXISTS demo_account_v;
CREATE VIEW demo_account_v AS SELECT id, name, balance FROM demo_accounts;

CREATE TRIGGER demo_account_v_update
INSTEAD OF UPDATE ON demo_account_v
BEGIN
    UPDATE demo_accounts
    SET name = NEW.name, balance = NEW.balance
    WHERE id = NEW.id;
END;

-- write through the view:
UPDATE demo_account_v SET balance = 500 WHERE id = 1;
SELECT * FROM demo_accounts;

## Clean up

In [ ]:
%%sql
DROP TRIGGER IF EXISTS demo_account_v_update;
DROP VIEW IF EXISTS demo_account_v;
DROP TRIGGER IF EXISTS demo_no_negative;
DROP TRIGGER IF EXISTS demo_bal_audit;
DROP TABLE IF EXISTS demo_accounts;
DROP TABLE IF EXISTS demo_audit;
SELECT 'cleaned up' AS status;

## Practice

**✏️ Exercise 1.** Describe (in a comment) one real use for an AFTER INSERT trigger, then write a trigger skeleton that logs every new row inserted into a table `demo_t(id, val)` into `demo_log(msg)`.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_t; DROP TABLE IF EXISTS demo_log;
CREATE TABLE demo_t (id INTEGER PRIMARY KEY, val TEXT);
CREATE TABLE demo_log (msg TEXT);
CREATE TRIGGER demo_t_ins AFTER INSERT ON demo_t
BEGIN
    INSERT INTO demo_log (msg) VALUES ('inserted id=' || NEW.id);
END;
INSERT INTO demo_t (val) VALUES ('hello');
SELECT * FROM demo_log;

### ✅ Recap
Triggers automate reactions to data changes (audit, validation with `RAISE`,
derived data) using `NEW`/`OLD`; `INSTEAD OF` triggers make views writable. Use
them judiciously — hidden logic can surprise future readers.

**Next:** `25_query_patterns_and_recipes.ipynb`.